# 🔒 Day 02a: Encapsulation & Abstraction — Control & Simplify

---

## 🎯 What You'll Master Today
- Encapsulation — hiding internal state, controlling access
- Abstraction — hiding complexity, showing only what matters
- Python's access modifiers: public, _protected, __private
- Abstract Base Classes (ABC) — enforcing contracts
- Name mangling — what `__var` actually does

---

## 🎭 The Analogy

Think of a **car**:

- **Encapsulation** = The engine is sealed under the hood. You can't directly touch the pistons. You interact through the steering wheel, pedals, and dashboard.
- **Abstraction** = You press the gas pedal and the car accelerates. You don't need to know about fuel injection, spark timing, or crankshaft rotation.

**Encapsulation** = *bundling data + methods together and restricting direct access*  
**Abstraction** = *hiding complexity and exposing only relevant features*

They're related but different:
- Encapsulation is about **protection** (don't touch my internals)
- Abstraction is about **simplification** (don't worry about how it works)

---

## 📐 Encapsulation vs Abstraction

```
╔═══════════════════════════════════════════════════════════════════╗
║  ENCAPSULATION vs ABSTRACTION                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  ENCAPSULATION                     ABSTRACTION                   ║
║  ┌─────────────────────┐          ┌─────────────────────┐       ║
║  │ Bundle data + methods│          │ Hide HOW it works   │       ║
║  │ into a single unit  │          │ Show WHAT it does    │       ║
║  │                     │          │                     │       ║
║  │ Control access:     │          │ Interface:          │       ║
║  │  public / private   │          │  Abstract classes   │       ║
║  │  getters / setters  │          │  Abstract methods   │       ║
║  └─────────────────────┘          └─────────────────────┘       ║
║                                                                   ║
║  Focus: DATA PROTECTION            Focus: COMPLEXITY HIDING      ║
║  How: access modifiers             How: abstract classes/methods ║
║                                                                   ║
║  Example: private balance           Example: .send_email()       ║
║  with deposit()/withdraw()          hides SMTP, DNS, encoding    ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Python Access Modifiers
# ============================================
# Python doesn't have true private — it's by CONVENTION

class Employee:
    def __init__(self, name: str, salary: float, ssn: str):
        self.name = name           # PUBLIC — anyone can access
        self._salary = salary      # PROTECTED — "internal use, subclasses ok"
        self.__ssn = ssn           # PRIVATE — name mangled to _Employee__ssn
    
    def get_ssn_masked(self):
        """Controlled access to sensitive data."""
        return f"***-**-{self.__ssn[-4:]}"


emp = Employee("Sarthak", 1500000, "123-45-6789")

# Public — direct access
print(f"Name: {emp.name}")  # ✅ Works fine

# Protected — accessible but signals "internal"
print(f"Salary: {emp._salary}")  # ✅ Works, but IDEs warn you

# Private — name mangled
try:
    print(emp.__ssn)  # ❌ AttributeError!
except AttributeError as e:
    print(f"Can't access __ssn directly: {e}")

# But you CAN access it via mangled name (Python doesn't enforce!)
print(f"Mangled access: {emp._Employee__ssn}")  # ✅ Works — Python trusts you

# Proper way — use the method
print(f"Masked SSN: {emp.get_ssn_masked()}")

## 📋 Python Access Modifiers

```
╔══════════════╦════════════╦══════════════════════════════════════╗
║  Convention   ║  Access    ║  Notes                              ║
╠══════════════╬════════════╬══════════════════════════════════════╣
║  self.name    ║  Public    ║  Anyone can read/write              ║
║  self._name   ║  Protected ║  "Internal" — subclasses ok,        ║
║               ║            ║  external use discouraged           ║
║  self.__name  ║  Private   ║  Name mangled to _ClassName__name  ║
║               ║            ║  NOT truly private — just harder    ║
╚══════════════╩════════════╩══════════════════════════════════════╝
```

> **Interview Answer:** "Python uses conventions, not enforcement — single underscore means 'internal use', double underscore triggers name mangling to prevent accidental access in subclasses. We're all consenting adults."

---

In [ ]:
# ============================================
# 2. Encapsulation with @property
# ============================================

class BankAccount:
    """Encapsulated account — balance can't be set directly."""
    
    def __init__(self, owner: str, initial_balance: float = 0):
        self._owner = owner
        self._balance = initial_balance
        self._transaction_log = []
    
    @property
    def balance(self):
        """Read-only property — no setter means can't assign directly."""
        return self._balance
    
    @property
    def owner(self):
        return self._owner
    
    def deposit(self, amount: float) -> 'BankAccount':
        if amount <= 0:
            raise ValueError("Amount must be positive")
        self._balance += amount
        self._transaction_log.append(f"DEPOSIT: +₹{amount:,.2f}")
        return self
    
    def withdraw(self, amount: float) -> 'BankAccount':
        if amount <= 0:
            raise ValueError("Amount must be positive")
        if amount > self._balance:
            raise ValueError(f"Insufficient funds (balance: ₹{self._balance:,.2f})")
        self._balance -= amount
        self._transaction_log.append(f"WITHDRAW: -₹{amount:,.2f}")
        return self
    
    def get_statement(self) -> str:
        header = f"\n--- Statement for {self._owner} ---"
        txns = "\n".join(self._transaction_log) or "No transactions"
        footer = f"Current balance: ₹{self._balance:,.2f}"
        return f"{header}\n{txns}\n{footer}\n"


acc = BankAccount("Sarthak", 10000)
acc.deposit(5000).withdraw(2000).deposit(1000)

print(f"Balance: ₹{acc.balance:,.2f}")  # ✅ Read via property

try:
    acc.balance = 999999  # ❌ No setter — can't cheat!
except AttributeError as e:
    print(f"Can't set balance directly: {e}")

print(acc.get_statement())

In [ ]:
# ============================================
# 3. Abstraction with ABC (Abstract Base Class)
# ============================================
from abc import ABC, abstractmethod

class Shape(ABC):
    """Abstract class — defines a CONTRACT.
    You CANNOT instantiate this directly.
    Subclasses MUST implement all @abstractmethod methods."""
    
    @abstractmethod
    def area(self) -> float:
        """Every shape must calculate its area."""
        pass
    
    @abstractmethod
    def perimeter(self) -> float:
        """Every shape must calculate its perimeter."""
        pass
    
    def describe(self) -> str:
        """Concrete method — shared by all shapes."""
        return f"{self.__class__.__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"


class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius
    
    def area(self) -> float:
        return 3.14159 * self.radius ** 2
    
    def perimeter(self) -> float:
        return 2 * 3.14159 * self.radius


class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height
    
    def area(self) -> float:
        return self.width * self.height
    
    def perimeter(self) -> float:
        return 2 * (self.width + self.height)


# Can't instantiate abstract class
try:
    s = Shape()
except TypeError as e:
    print(f"Can't instantiate ABC: {e}")

# Concrete classes work fine
shapes = [Circle(5), Rectangle(4, 6)]
for shape in shapes:
    print(shape.describe())  # polymorphism! same method, different behavior

In [ ]:
# ============================================
# 4. What happens if you DON'T implement abstract methods?
# ============================================

class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height
    
    def area(self) -> float:
        return 0.5 * self.base * self.height
    
    # Oops! Forgot to implement perimeter()

try:
    t = Triangle(3, 4)  # ❌ Error! perimeter() not implemented
except TypeError as e:
    print(f"Contract violation: {e}")

# This is the POWER of ABCs — catch missing implementations at instantiation,
# not at runtime when the method is called.

In [ ]:
# ============================================
# 5. Real-World Abstraction — Payment System
# ============================================
from abc import ABC, abstractmethod

class PaymentProcessor(ABC):
    """Define WHAT any payment processor must do.
    Don't care HOW — that's up to each implementation."""
    
    @abstractmethod
    def process_payment(self, amount: float) -> bool:
        pass
    
    @abstractmethod
    def refund(self, transaction_id: str, amount: float) -> bool:
        pass
    
    def validate_amount(self, amount: float) -> bool:
        """Shared validation — concrete method."""
        return amount > 0


class StripeProcessor(PaymentProcessor):
    def process_payment(self, amount: float) -> bool:
        if not self.validate_amount(amount):
            return False
        print(f"[Stripe] Charging ₹{amount:,.2f} via Stripe API")
        return True
    
    def refund(self, transaction_id: str, amount: float) -> bool:
        print(f"[Stripe] Refunding ₹{amount:,.2f} for txn {transaction_id}")
        return True


class RazorpayProcessor(PaymentProcessor):
    def process_payment(self, amount: float) -> bool:
        if not self.validate_amount(amount):
            return False
        print(f"[Razorpay] Charging ₹{amount:,.2f} via Razorpay API")
        return True
    
    def refund(self, transaction_id: str, amount: float) -> bool:
        print(f"[Razorpay] Refunding ₹{amount:,.2f} for txn {transaction_id}")
        return True


# Client code doesn't care WHICH processor — abstraction!
def checkout(processor: PaymentProcessor, amount: float):
    """Works with ANY PaymentProcessor — depends on abstraction."""
    if processor.process_payment(amount):
        print("Payment successful! ✅")
    else:
        print("Payment failed! ❌")


checkout(StripeProcessor(), 999)
print()
checkout(RazorpayProcessor(), 1499)

In [ ]:
# ============================================
# 6. Name Mangling Deep Dive
# ============================================

class Parent:
    def __init__(self):
        self.__secret = "parent_secret"  # mangled to _Parent__secret
        self._protected = "parent_protected"

class Child(Parent):
    def __init__(self):
        super().__init__()
        self.__secret = "child_secret"  # mangled to _Child__secret
        # Does NOT override parent's __secret — different mangled names!

c = Child()

# Both exist on the object with different mangled names
print(f"Child's __secret:  {c._Child__secret}")   # child_secret
print(f"Parent's __secret: {c._Parent__secret}")  # parent_secret

# This is WHY name mangling exists:
# To prevent accidental name collisions in inheritance hierarchies.
# It's NOT about security — it's about avoiding bugs.

print(f"\nAll attributes: {[a for a in dir(c) if 'secret' in a]}")

## 🗺️ Encapsulation & Abstraction Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  ENCAPSULATION & ABSTRACTION — QUICK REFERENCE                   ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  ENCAPSULATION:                                                   ║
║    public (name) → _protected → __private (name mangled)        ║
║    Use @property for controlled access                           ║
║    Python = conventions, not enforcement                         ║
║                                                                   ║
║  ABSTRACTION:                                                     ║
║    ABC + @abstractmethod = define contracts                      ║
║    Can't instantiate ABC directly                                ║
║    Missing implementation → TypeError at instantiation           ║
║    Concrete methods in ABC = shared behavior                     ║
║                                                                   ║
║  NAME MANGLING:                                                   ║
║    __var → _ClassName__var                                       ║
║    Purpose: prevent collision in inheritance, NOT security        ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions — Practice These

| # | Question | Key Point |
|---|----------|------------|
| 1 | Encapsulation vs Abstraction? | Encapsulation = data protection (hide state). Abstraction = complexity hiding (hide implementation) |
| 2 | How does Python implement private? | Name mangling: __var → _ClassName__var. Convention-based, not enforced |
| 3 | What is an ABC? | Abstract Base Class — defines a contract. Can't instantiate, subclasses must implement abstract methods |
| 4 | Why use @property? | Pythonic getters/setters. Access like attribute, but runs method for validation/computation |
| 5 | _ vs __ in Python? | _ = convention for internal. __ = name mangling to prevent collision in inheritance |
| 6 | Can you access __var outside the class? | Yes, via _ClassName__var. Python trusts developers — "we're all consenting adults" |
| 7 | Abstract class vs Interface? | Python ABCs can have both abstract and concrete methods. Pure interfaces (Protocol) have only abstract methods |
| 8 | When to use ABC? | When you want to enforce a contract — every subclass MUST implement certain methods |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Encapsulation = bundle data + control access          │
## │  • Abstraction = hide complexity, show interface         │
## │  • Python: public, _protected, __private (mangled)      │
## │  • @property = Pythonic getters/setters                  │
## │  • ABC + @abstractmethod = enforce contracts             │
## │  • Name mangling prevents collision, NOT security        │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Inheritance** — code reuse, method overriding, and the MRO

# 🧠 Encapsulation & Abstraction — Day 2A

Two pillars of OOP that interviewers LOVE to mix up. Let's nail both.

## 🧠 Encapsulation — The Gist

**Bundle data + methods together**, control who can touch what.

Think of it like a **car dashboard**:
- You see: speedometer, fuel gauge, steering wheel
- You DON'T see: engine internals, fuel injection, transmission

```
┌─────────────────────────────────┐
│          CAR (Object)           │
│  ┌───────────────────────────┐  │
│  │   Dashboard (Public API)  │  │
│  │  • start()  • brake()    │  │
│  │  • accelerate()          │  │
│  └───────────────────────────┘  │
│  ┌───────────────────────────┐  │
│  │   Engine (Private/Hidden) │  │
│  │  • __fuel_injection()    │  │
│  │  • __ignite_spark()      │  │
│  │  • __rpm = 3000          │  │
│  └───────────────────────────┘  │
└─────────────────────────────────┘
```

👉 **Why?** Prevents outside code from messing up internal state. Change internals without breaking external code.

In [ ]:
# Basic encapsulation — bundling data + behavior
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner        # public
        self.__balance = balance  # private (name-mangled)
    
    def deposit(self, amount):
        if amount > 0:
            self.__balance += amount
            return f"Deposited {amount}. Balance: {self.__balance}"
        return "Invalid amount"
    
    def get_balance(self):  # controlled access
        return self.__balance

acc = BankAccount("Sarthak", 1000)
print(acc.deposit(500))
print(acc.get_balance())
# print(acc.__balance)  # ❌ AttributeError!

## 🔐 Access Modifiers in Python

Python doesn't have TRUE private — it's all **convention + name mangling**.

```
┌──────────────┬────────────┬──────────────────────────┐
│  Convention  │  Prefix    │  Meaning                 │
├──────────────┼────────────┼──────────────────────────┤
│  Public      │  name      │  Anyone can access       │
│  Protected   │  _name     │  "Please don't touch"    │
│  Private     │  __name    │  Name-mangled by Python  │
└──────────────┴────────────┴──────────────────────────┘
```

In [ ]:
class Employee:
    def __init__(self, name, salary, ssn):
        self.name = name          # public — go ahead, use it
        self._salary = salary     # protected — "hey, be careful"
        self.__ssn = ssn          # private — name mangled

emp = Employee("Alice", 90000, "123-45-6789")

print(emp.name)       # ✅ Works fine
print(emp._salary)    # ✅ Works, but you're being naughty
# print(emp.__ssn)    # ❌ AttributeError

# 🔍 Name mangling — Python renames __attr to _ClassName__attr
print(emp._Employee__ssn)  # ✅ Gotcha! Still accessible
print(dir(emp))            # See _Employee__ssn in there

In [ ]:
# Name mangling in inheritance — this is where it gets spicy 🌶️
class Base:
    def __init__(self):
        self.__secret = "base secret"
    
    def reveal(self):
        return self.__secret  # works — same class

class Child(Base):
    def try_access(self):
        # return self.__secret  # ❌ Looks for _Child__secret, not _Base__secret
        return self._Base__secret  # ✅ Must use mangled name

c = Child()
print(c.reveal())       # works via Base's method
print(c.try_access())   # works via mangled name

# 👉 Key insight: name mangling prevents ACCIDENTAL override in subclasses
# It's NOT a security feature — it's a namespace collision preventer

## 🎯 Property Decorators Deep Dive

The Pythonic way to do getters/setters — no ugly `get_x()` / `set_x()` methods.

```
  @property        →  getter  (obj.x)
  @x.setter        →  setter  (obj.x = val)
  @x.deleter       →  deleter (del obj.x)
```

In [ ]:
class Temperature:
    """Temperature in Kelvin — can't go below 0K (absolute zero)"""
    
    def __init__(self, kelvin=0):
        self.kelvin = kelvin  # goes through setter!
    
    @property
    def kelvin(self):
        return self._kelvin
    
    @kelvin.setter
    def kelvin(self, value):
        if value < 0:
            raise ValueError("Temperature below absolute zero? Physics says no. 🚫")
        self._kelvin = value
    
    @kelvin.deleter
    def kelvin(self):
        print("Resetting temperature to 0K")
        self._kelvin = 0
    
    @property
    def celsius(self):  # computed property — read-only
        return self._kelvin - 273.15
    
    @property
    def fahrenheit(self):
        return self._kelvin * 9/5 - 459.67

t = Temperature(300)
print(f"K: {t.kelvin}, C: {t.celsius:.1f}, F: {t.fahrenheit:.1f}")

t.kelvin = 373.15
print(f"Boiling water: {t.celsius:.1f}°C")

del t.kelvin
print(f"After delete: {t.kelvin}K")

try:
    t.kelvin = -10  # 💥
except ValueError as e:
    print(e)

👉 **Key insight**: `@property` lets you start with simple attributes and add validation LATER without changing the API. That's encapsulation done right.

## 🧠 Abstraction — The Gist

**Hide the HOW, show the WHAT.**

Think of a **TV remote**:
- You press `volume_up()` — simple
- Behind the scenes: IR signal → receiver → DAC → amplifier → speaker voltage adjustment
- You don't care about ANY of that

```
┌─────────────────────────────────────────┐
│            ABSTRACTION                  │
│                                         │
│   What you see:    press_button()       │
│                        │                │
│   ─ ─ ─ ─ ─ ─ ─ ─ ─ ─│─ ─ ─ ─ ─ ─    │
│                        ▼                │
│   Hidden:     encode_ir_signal()        │
│               transmit_signal()         │
│               decode_at_receiver()      │
│               adjust_voltage()          │
│               update_display()          │
└─────────────────────────────────────────┘
```

👉 **Abstraction = defining WHAT** an object does (interface)  
👉 **Encapsulation = controlling HOW** it does it (implementation hiding)

## 📐 Abstract Base Classes (ABC)

Force subclasses to implement certain methods — or they can't be instantiated.

In [ ]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    """Abstract base — can't instantiate directly"""
    
    @abstractmethod
    def area(self) -> float:
        pass
    
    @abstractmethod
    def perimeter(self) -> float:
        pass
    
    def describe(self):  # concrete method — shared by all
        return f"{self.__class__.__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    
    def area(self):
        return math.pi * self.radius ** 2
    
    def perimeter(self):
        return 2 * math.pi * self.radius


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height
    
    def area(self):
        return self.width * self.height
    
    def perimeter(self):
        return 2 * (self.width + self.height)


# Can't do this:
try:
    s = Shape()  # 💥
except TypeError as e:
    print(f"❌ {e}")

# But these work:
shapes = [Circle(5), Rectangle(4, 6)]
for s in shapes:
    print(s.describe())

In [ ]:
# What if you forget to implement an abstract method?
class Triangle(Shape):
    def __init__(self, a, b, c):
        self.a, self.b, self.c = a, b, c
    
    def perimeter(self):
        return self.a + self.b + self.c
    # forgot area()! 😬

try:
    t = Triangle(3, 4, 5)
except TypeError as e:
    print(f"❌ {e}")
    # Python catches this at INSTANTIATION, not at class definition
    # 👉 Interview gotcha: the error is at instantiation, not definition!

## 🦆 Protocols (Structural Typing) — Python 3.8+

Duck typing, but **formalized with type hints**. No inheritance needed.

"If it walks like a duck and quacks like a duck..." → now the type checker agrees too.

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Drawable(Protocol):
    def draw(self) -> str: ...  # just the signature

# No inheritance from Drawable!
class Button:
    def draw(self) -> str:
        return "[  Click Me  ]"

class TextBox:
    def draw(self) -> str:
        return "[____________]"

class NotDrawable:
    def render(self) -> str:  # wrong method name
        return "nope"

def render_all(widgets: list[Drawable]):
    for w in widgets:
        print(w.draw())

render_all([Button(), TextBox()])  # ✅ works — structural match

# Runtime check:
print(isinstance(Button(), Drawable))      # True
print(isinstance(NotDrawable(), Drawable))  # False

# 👉 Protocol = ABC but WITHOUT forced inheritance
# 👉 Prefer Protocol when you want loose coupling

## ⚔️ Encapsulation vs Abstraction

The **#1 interview confusion**. Here's the cheat sheet:

```
┌──────────────────┬───────────────────────┬───────────────────────┐
│                  │   ENCAPSULATION       │   ABSTRACTION         │
├──────────────────┼───────────────────────┼───────────────────────┤
│  Focus           │  HOW (implementation) │  WHAT (interface)     │
│  Purpose         │  Hide internal state  │  Hide complexity      │
│  Mechanism       │  Access modifiers,    │  ABC, Protocol,       │
│                  │  properties, closures │  interfaces           │
│  Level           │  Object level         │  Design level         │
│  Example         │  Private __balance    │  Abstract area()      │
│                  │  with deposit()       │  in Shape class       │
│  Analogy         │  Car engine is hidden │  Steering wheel is    │
│                  │  behind the hood      │  simple to use        │
│  Key Question    │  "Who can access      │  "What must every     │
│                  │   this data?"         │   subclass do?"       │
└──────────────────┴───────────────────────┴───────────────────────┘
```

👉 **TL;DR**: Encapsulation = data hiding. Abstraction = complexity hiding.  
👉 They work TOGETHER — encapsulation is often the mechanism that achieves abstraction.

## 🗺️ Summary Map

```
┌───────────────── ENCAPSULATION ──────────────────┐
│                                                   │
│   public (name)          →  anyone can access     │
│   _protected (_name)     →  convention only       │
│   __private (__name)     →  name-mangled          │
│                              _Class__name         │
│                                                   │
│   @property              →  Pythonic getters      │
│   @x.setter              →  validation on set     │
│   @x.deleter             →  cleanup on delete     │
│                                                   │
└───────────────────────────────────────────────────┘
                      │
                      ▼
┌──────────────── ABSTRACTION ─────────────────────┐
│                                                   │
│   ABC + @abstractmethod  →  enforce interface     │
│   Protocol               →  structural typing     │
│                              (duck typing + hints) │
│                                                   │
│   Concrete methods in    →  shared behavior       │
│   abstract classes          (template pattern)    │
│                                                   │
│   Error at INSTANTIATION →  not at definition!    │
│                                                   │
└───────────────────────────────────────────────────┘
```

## 💡 Interview Quick-Fire

**Q1: What's encapsulation?**  
A: Bundling data and methods into a single unit (class) and restricting direct access to internal state. Controls HOW data is accessed/modified.

**Q2: Does Python have true private variables?**  
A: No. `__name` uses name mangling (`_ClassName__name`) — it's accessible, just harder to reach accidentally. It's a convention, not enforcement.

**Q3: What's the difference between `_x` and `__x`?**  
A: `_x` is a convention ("protected — don't touch"). `__x` triggers name mangling — Python renames it to `_ClassName__x` to prevent accidental access in subclasses.

**Q4: What's abstraction vs encapsulation?**  
A: Abstraction hides COMPLEXITY (what to do — interface). Encapsulation hides DATA (how it's done — implementation). Encapsulation is a mechanism to achieve abstraction.

**Q5: Can an abstract class have concrete methods?**  
A: Yes! Abstract classes can mix abstract methods (must override) with concrete methods (shared implementation). This is the Template Method pattern.

**Q6: When does Python raise an error for unimplemented abstract methods?**  
A: At **instantiation**, not at class definition. You can define an incomplete subclass — you just can't create an instance of it.

**Q7: ABC vs Protocol — when to use which?**  
A: Use ABC when you control the hierarchy and want forced inheritance. Use Protocol when you want structural typing (duck typing with type hints) — no inheritance required. Protocol = loose coupling.

**Q8: Why use `@property` instead of `get_x()` / `set_x()`?**  
A: Pythonic — access looks like an attribute (`obj.x`), but runs validation logic behind the scenes. You can start with a plain attribute and add `@property` later WITHOUT changing the API.